# Spark DataFrames: a distributed table with a schema

_Apache Spark_

**The modern, optimized Spark API — like a pandas DataFrame, but lazy, distributed, and rewritten into an efficient plan by the Catalyst optimizer.**

A Spark DataFrame is a distributed table with a schema: rows of typed, named
       columns, physically split into partitions spread across the machines of a cluster. On the surface
       it looks just like a pandas DataFrame &mdash; you select columns, filter
       rows, add columns &mdash; but two deep differences change how you use it.

       
         
- It is lazy. When you write df.filter(...).select(...), nothing runs. Spark
         just records what you asked for, building a plan. Only an action &mdash;
         show, count, collect, write &mdash; triggers
         actual computation. (pandas, by contrast, is eager: each line runs immediately.)
         
- It is optimized. Before running, the plan is handed to the Catalyst optimizer,
         which rewrites it &mdash; reordering filters, pruning unused columns, pushing predicates down into
         the file read, collapsing steps &mdash; into an efficient physical plan. You write what you
         want; Spark figures out a good how.
       

---

This notebook is a practice scaffold for the **Spark DataFrames: a distributed table with a schema** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q pyspark
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_iris

data = load_iris(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — PySpark

In [ ]:
# Colab: !pip install pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# 1. A SparkSession: your handle to the engine. local[*] = use all cores of this one Colab VM.
spark = (SparkSession.builder
         .master("local[*]")
         .appName("dataframes-demo")
         .getOrCreate())

# 2. Build a DataFrame with an EXPLICIT schema (no inference -> fast + exact types).
schema = StructType([
    StructField("name", StringType(), nullable=False),
    StructField("age",  IntegerType(), nullable=False),
    StructField("city", StringType(), nullable=False),
])
rows = [
    ("Ada",   36, "London"),
    ("Linus", 25, "Helsinki"),
    ("Grace", 41, "New York"),
    ("Alan",  29, "London"),
    ("Ada",   36, "London"),   # a duplicate row, on purpose
]
df = spark.createDataFrame(rows, schema)

# 3. LOOK at it (printSchema + show + describe are how you inspect a DataFrame).
df.printSchema()          # column names and types -- NOT print(df)
df.show()                 # an ACTION: prints the rows. use .show(), never print(df)
df.describe().show()      # count / mean / stddev / min / max per column

# 4. TRANSFORM it. Each call returns a NEW DataFrame (immutable); nothing runs yet (lazy).
result = (
    df.distinct()                                   # drop the duplicate row
      .filter(F.col("age") > 30)                    # .where() is an exact alias of .filter()
      .withColumn("dog_years", F.col("age") * 7)    # add a computed column via a column expression
      .drop("city")                                 # remove a column we no longer need
      .select("name", "age", "dog_years")           # keep just these columns
)

# 5. INSPECT THE OPTIMIZED PLAN, then TRIGGER it.
result.explain()          # shows Catalyst's physical plan (filter pushed down, 'city' pruned)
result.show()             # the ACTION that actually runs the optimized query and prints rows
print("matching rows:", result.count())   # count() is also an action

# Bonus: the SAME query as plain SQL compiles to the SAME optimized plan.
df.createOrReplaceTempView("people")
spark.sql("SELECT name, age, age * 7 AS dog_years FROM people WHERE age > 30").show()

spark.stop()

## Visualize the data & results

_When a Spark DataFrame query filters then counts per group, what comes back? Illustrated with the equivalent pandas query on the real Iris dataset: filter petal length > 1.5 cm, then count rows per species._

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris

# Real 150-row Iris dataset. This pandas computation ILLUSTRATES what the
# Spark DataFrame query  df.filter(F.col("petal_length") > 1.5)
#                          .groupBy("species").count()  returns.
d = load_iris(as_frame=True)
df = d.frame.rename(columns={"petal length (cm)": "petal_length"})
df["species"] = df["target"].map({0: "setosa", 1: "versicolor", 2: "virginica"})

all_counts = df.groupby("species").size()                    # before the filter
filtered   = df[df["petal_length"] > 1.5]                     # the .filter(...) step
flt_counts = filtered.groupby("species").size()              # the .groupBy().count() step

for sp in ["setosa", "versicolor", "virginica"]:
    print(f"{sp:11s} all {all_counts[sp]:3d}  ->  filtered {flt_counts[sp]:3d}")
# setosa      all  50  ->  filtered  13
# versicolor  all  50  ->  filtered  50
# virginica   all  50  ->  filtered  50
# In Spark you'd see this with result.show(), not print -- and it would be
# built lazily and computed across partitions on a cluster.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You write df2 = df.filter(F.col("amount") > 100).select("user", "amount") and then print(df2), but you see something like DataFrame[user: string, amount: double] instead of any rows, and it returned instantly. What happened, and how do you actually see the data and the plan?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recognize that filter and select are transformations &mdash; lazy. They built a plan; they ran nothing. — _A Spark DataFrame is lazy: transformations only record intent. That is why it returned instantly._
- Recognize that print(df2) prints the DataFrame's repr (its schema), not its rows. — _print is not a Spark action; it never triggers computation, so no rows are fetched._
- Call an action to run it: df2.show() for the rows. Use df2.explain() to see the optimized plan. — _show() is an action &mdash; it runs the optimized query and prints rows; explain() reveals Catalyst's physical plan._

**Answer:** Nothing ran. filter and select are lazy transformations &mdash; they only built a query plan, which is why the call returned instantly. print(df2) shows the DataFrame's schema repr, not its data, because print is not a Spark action. To see the rows, call an action: df2.show(). To see the optimized plan Catalyst built (e.g. the amount > 100 filter pushed down), call df2.explain(). This is the number-one pandas-habit trap in Spark.

</details>

**Problem 2.** A teammate has a function clean(s) in Python and wraps it as a UDF (User-Defined Function) to uppercase and trim a city column over a billion-row DataFrame. The job is painfully slow. What is the likely cause and the fix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Identify that a Python UDF runs row-by-row in a separate Python process, outside the JVM engine. — _Every row is serialized out to Python and back, and the Catalyst optimizer cannot see inside the UDF, so it cannot optimize it._
- Check whether built-in pyspark.sql.functions can do the same job: here F.trim(F.upper(F.col("city"))). — _Built-in F. functions run inside the engine, stay vectorized, and are visible to the optimizer._
- Replace the UDF: df.withColumn("city", F.trim(F.upper(F.col("city")))). — _No per-row Python round-trip; the operation is pushed into the optimized plan and runs far faster._

**Answer:** The slowness is the Python UDF: it ships every one of the billion rows out to a Python process and back, and the Catalyst optimizer cannot see inside it, so it gets no optimization. The fix is to express the same logic with built-in pyspark.sql.functions &mdash; here df.withColumn("city", F.trim(F.upper(F.col("city")))). Built-in F. functions run inside the engine and are optimizable, so they are dramatically faster. Only reach for a UDF (and prefer a vectorized/pandas UDF) when no built-in can express the logic.

</details>

**Problem 3.** You read a big CSV of orders with spark.read.csv(path, header=True, inferSchema=True). It is slow to read, and a column of order IDs like "007" and "012" comes back as integers 7 and 12, losing the leading zeros. Diagnose both problems and give the fix.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recognize inferSchema=True makes Spark scan the file an extra time to guess column types. — _That second pass over a big file is the source of the slow read._
- Recognize the order-ID column is really a string identifier, but inference saw all-digits and guessed integer, dropping the leading zeros. — _Schema inference guesses from the data and gets identifier-like columns wrong &mdash; an integer cannot hold "007"._
- Define an explicit StructType with order_id as StringType() and pass schema=... instead of inferSchema. — _An explicit schema reads in one pass (faster) and forces the correct types, so "007" stays a string._

**Answer:** Both problems come from schema inference. The slow read is the extra data scan inferSchema=True does to guess types. The corrupted IDs are inference guessing wrong: an all-digits column looked like an integer, so "007" became 7 and the leading zeros were lost (an integer cannot store them). The fix is an explicit StructType schema with order_id typed as StringType(), passed via schema=.... It reads in a single pass (faster) and pins the right types &mdash; the standard practice for production reads.

</details>